[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeremydwong/bonelab_inverse_dynamics/blob/main/notebooks/aff_bowed_femur.ipynb)

# Bowed femurs and atypical femoral fracture: what bowing our walkers does to knee and femoral-shaft loads

Atypical femoral fracture (AFF) is a stress fracture that starts in the lateral cortex of the femoral shaft. It's much more common in East and Southeast Asian women, and the suspicion (mine, and a good part of the literature's) is that leg geometry is part of the reason: a femur that bows laterally puts its outer cortex further into tension every step. So this page does three things. It reads the literature, it takes real walkers from the van der Zee, Mundinger & Kuo (2022) dataset and bends their legs on the computer, and then it speculates a little about whether walking differently could help.

> **How much to trust this page.** The loads come from the same pipeline as the rest of these reports (the walking data validated against Visual3D in chapter 4). What's new is a *counterfactual*: same gait, different skeleton. Real people with bowed legs walk a bit differently, and the numbers here are net joint loads with no muscles. So read them as "how much does geometry alone change the load", not as a prediction for a patient.

This notebook reports on the following key findings about our reanalysis:

1. **The literature agrees AFF is a tension-side fatigue fracture whose location follows the bow.** The 2014 ASBMR task force calls AFFs stress fractures of the lateral cortex. In Korean women the fractured femurs bowed laterally 12.4° vs 4.0° in controls (Jang 2017), the fracture sits at the apex of the bow (within 4.3 ± 3.3 % of femur length, Park 2019), and Asian women carry a 4.84× hazard vs white women on bisphosphonates (Black 2020). Lateral bowing in Asian women is mostly *acquired* with age and low bone density, not a population constant.
2. **Moving the knee laterally (varus) raises the knee adduction moment by about 7 % per degree.** On p6 at 1.25 m/s the first peak goes 0.510 → 0.702 N m/kg for a 20 mm lateral knee shift (5.2° of added varus). Across all nine walkers it's 5.6 to 10.8 % per degree (median 7.3 %).
3. **Bowing the femoral shaft raises the lateral-tension bending moment at mid-shaft by about 10 % per 10 mm of bow.** On p6, 0.995 → 1.187 N m/kg for a 0 → 20 mm bow (9.4 to 13.9 % per 10 mm across the nine walkers) (the literature puts controls near 4–6 mm and diaphyseal AFF femurs near 10–20 mm). The added moment is the axial force times the offset (ΔM ≈ P·δ), and in vivo the axial force is 2–3× our net value, so the real increment is probably bigger.
4. **Wider steps help a little.** p6 walked at measured widths of 10.8 to 34.6 cm. Going from their own ~16.5 cm to 34.6 cm lowered the knee adduction moment peak by 10–14 % and mid-shaft bending by 6–8 %, which offsets roughly 6–8 mm of bow. Nobody has tested a gait change against femoral strain or AFF, so this is new (but it's one subject, and run-to-run variation is about as big as the difference between neighbouring widths).

Chapter 6, what-if studies on our walkers. Previous: [the Asics reanalysis](asics_running_c3d.ipynb).

In [1]:
# Colab bootstrap: a no-op when running locally. On Colab it clones the repo (for its
# data/ folder), installs boneid from the clone, and runs from notebooks/ so the
# relative paths below work.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("../src/boneid"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/jeremydwong/bonelab_inverse_dynamics"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "bonelab_inverse_dynamics"], check=True)
    os.chdir("bonelab_inverse_dynamics/notebooks")
os.makedirs("../reports", exist_ok=True)

## 1. What the literature says

I asked three questions of the literature: what is the mechanism of AFF, how does leg geometry differ in East and Southeast Asian women, and how does a varus (bowed) leg change walking loads. Every paper below was checked to exist by DOI, and the citation counts (Semantic Scholar, October 2026) are in the [references](#References). I've mostly read abstracts, so I quote numbers the way the abstracts give them.

### 1a. AFF is a tensile stress fracture of the lateral cortex

* **What it is.** The ASBMR task force defined AFF in 2010 (Shane 2010) and revised it in 2014 (Shane 2014): a transverse or short-oblique fracture of the subtrochanteric region or femoral shaft, with minimal trauma, and a periosteal reaction on the *lateral* cortex. The 2014 revision calls them stress or insufficiency fractures. Schilcher 2013 found the radiographic signature is very specific: a transverse fracture angle of 89° ± 10° and a lateral callus. A transverse crack starting on one side is what a fatigue crack in bending looks like.
* **Bisphosphonates matter a lot, and so does how long you take them.** Relative risk with bisphosphonate use was 47.3 in Sweden (Schilcher 2011), and risk fell about 70 % per year after stopping. In the large Kaiser cohort the hazard ratio vs < 3 months of use went from 8.86 at 3–5 years to 43.51 at ≥ 8 years (Black 2020).
* **Walking is the worst everyday activity for the lateral shaft.** In a musculoskeletal + finite-element model, lateral-shaft tensile strain ranged from 327 µε (squat) to 2004 µε (walking), and joint moments plus ground force explained R² = 0.60 of it proximally (Martelli 2014). That's the link that makes a gait analysis relevant at all.
* **Geometry changes strain under the same loads.** Morphing ten CT-based femurs, shaft diameter, lateral bow and anterior bow had the largest effects on peak strain (Haider 2018). In Japanese CT finite-element models, bowed femurs that went on to stress fracture had higher peak tensile stress on the anterolateral surface, with no difference in bone density (Oh 2014b).

### 1b. The bow sets where the fracture happens

* **Lateral bow → mid-shaft; varus neck → subtrochanteric.** Mid-shaft AFFs have more lateral and anterior bow; subtrochanteric AFFs have a more varus (smaller) neck-shaft angle (Oh 2017, Hagen 2014). In Oh 2017's finite-element models the peak tensile stress sat next to the actual fracture, and the ratio of mid-shaft to subtrochanteric stress correlated with lateral bow (r = 0.64).
* **The fracture sits at the apex.** In bowed Korean femurs the fracture was 4.3 ± 3.3 % of femur length from the apex of the bow, mostly at 40–60 % of length from the greater trochanter (Park 2019). A lateral bow above about 5.25° separated diaphyseal from subtrochanteric sites (Yoo 2017).
* **The whole limb matters too.** Fracture site correlated with standing femorotibial angle (r = 0.82): the more varus the leg, the more distal the fracture (Saita 2014).
* **Muscles may be part of it.** AFF patients had less thigh muscle as well as more bow (Shin 2017), which the authors read as a failed "tension band". That idea comes back in section 2.

### 1c. East and Southeast Asian women: more risk, and more (acquired) bow

* **Risk.** Hazard ratio Asian vs white women on bisphosphonates: 4.84 (Black 2020). Incidence 64.2 vs 7.6 per 100,000 person-years, and adjusting for how long they took the drug barely changed it (Lo 2016). Hip bone density wasn't associated with AFF risk (Black 2022), which points away from density and towards geometry.
* **Where.** Japanese series are mostly mid-shaft (25 of 38 in the middle third, Hyodo 2017), whereas Western series are mostly subtrochanteric. About half of AFFs in a Singapore series had no bisphosphonate exposure (Gani 2022), and bowed-femur stress fractures happen without the drug at all (Oh 2014a).
* **The bow is mostly acquired.** I expected "Asian women have bow legs" to be true from the start. It mostly isn't:
  * Young adults are close to straight. Mean coronal bow is about 0° in an unselected Japanese CT cohort (Shimosawa 2019), and young Korean and Japanese women are only 1.35° and 2.0° varus at the hip-knee-ankle angle (Song 2015, Wanezaki 2021). Constitutional varus (≥ 3°) is 20–35 % in Korean and Japanese women vs 17 % in Belgian women (Bellemans 2012), so a bit more, not a lot.
  * Lateral bow grows with age, low bone density, low vitamin D and squatting work. It's present in most older Korean women (Lasam 2013), and Japanese women with the most curved femurs are older with less vitamin D (Tsuchie 2016).
  * Anterior bow differences are mostly femur length: Asian women have the most curved femurs in absolute terms because their femurs are shorter (Maratt 2014, Thiesen 2018).

So the numbers I'll use: controls and subtrochanteric AFF ≈ 3–4° of lateral bow, diaphyseal AFF ≈ 10–12° (Yoo 2017, Jang 2017). Over a ~360 mm shaft that's an apex offset of roughly 4–6 mm for controls and 10–20 mm for diaphyseal AFF (my geometry, from bow angle and shaft length).

### 1d. Bowed legs and walking loads

* **Varus raises the knee adduction moment.** Mechanical axis alignment correlates with peak knee adduction moment (R = 0.74, Hurwitz 2002); each degree of varus adds about 5 % to the medial share of knee load in vivo (Halder 2012); and varus raises the odds of medial osteoarthritis progression (Sharma 2001).
* **Muscles turn shaft bending into compression.** With all the thigh muscles included, the femur is "loaded axially, rather than in bending" (Duda 1997). The iliotibial band and adductors are needed to get realistic shaft strain (Duda 1998). In vivo, the proximal lateral femur showed tensile axial strain in stance (Aamodt 1997). So any bending number from inverse dynamics alone (like mine below) is the *net* load the muscles have to deal with, not the bone strain.
* **Gait changes that cut frontal-plane loads.** Most of this literature is about knee osteoarthritis. Lateral trunk sway cut the knee adduction moment by 65 % and hip adduction moment by 57.1 % (Mündermann 2008); toe-in gait cut the first peak by 13 % (Shull 2013); a contralateral cane by 10 % (Kemp 2008). The most direct hip-level evidence: in muscle-driven simulations "a wide base gait pattern has the largest potential to reduce hip contact forces" (Wesseling 2015). Careful, though: a lower knee adduction moment doesn't guarantee a lower medial contact force (Walter 2010).
* **The gap.** I couldn't find any study testing a gait change against femoral shaft strain or AFF. That's what section 3 tries, on one subject.

## 2. Bending our walkers' legs

### The data and the two counterfactuals

The walkers are from van der Zee, Mundinger & Kuo (2022): nine adults on a dual-belt treadmill, 5-stride Visual3D exports. I use p6 as the main subject (its marker-based alignment is near neutral, about 0°, while P1's markers show about 9° of apparent varus that I think is a joint-centre artefact), at the 1.25 m/s reference trial with self-selected step width. Right leg, same Newton–Euler recursion and settings as every other page here.

**A. Varus knee.** Every frame, I move the knee centre laterally by $d$, perpendicular to the hip–ankle line. The hip, ankle, foot and the *measured* ground force stay where they are, and the thigh and shank are re-posed by the smallest rotation. Then I re-run the inverse dynamics and read the external knee adduction moment (KAM, positive = varus). To first order this is just the ground force times the extra lever arm:

$$\Delta \text{KAM} \approx d \cdot F_z,\qquad \text{added varus} \approx d\left(\tfrac{1}{L_\text{thigh}} + \tfrac{1}{L_\text{shank}}\right)$$

**B. Bowed femoral shaft.** Here the joint centres don't move (a bowed shaft between the same hip and knee), so the joint loads are unchanged; what changes is *where the bone is* relative to those loads. At a cut at fraction $s$ along the femur (0 = knee, 1 = hip) I take the net internal wrench from the joint on the near side, and move it to the shaft centroid, which sits laterally by

$$e(s) = s\,L\tan\theta_{am} + 4\,\delta_0\, s(1-s)$$

($\theta_{am} \approx 6°$ is the normal angle between the anatomical and mechanical femoral axes; $\delta_0$ is the bow's apex offset). The frontal-plane bending moment is then $M_c = M_s + (q - c)\times F_s$, signed so positive means the **lateral cortex is in tension**. For a bow the extra bit is just the axial force times the offset, $\Delta M \approx P\,\delta$.

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display

from boneid import io_v3d as io
from boneid.core import SegmentKinematics, inverse_dynamics, chain_kinematics, detect_contact
from boneid.report import fig_svg, p1_params, JOINT_COLORS, style_axes

# The nine full Visual3D exports (~84 MB each) aren't in the repo. Point BONEID_V3D_DIR
# at a folder holding p1_5StridesData.mat ... p9_5StridesData.mat to run this elsewhere.
V3D_DIR = os.environ.get("BONEID_V3D_DIR", "/Users/jeremy/Dropbox/Public/inverse-dynamics-test-data")
DATA = os.path.join(V3D_DIR, "p{}_5StridesData.mat")
if not os.path.exists(DATA.format(6)):
    raise FileNotFoundError(
        f"This page needs the van der Zee, Mundinger & Kuo (2022) 5-stride exports for all nine "
        f"subjects; {DATA.format(6)} isn't there. Set BONEID_V3D_DIR to the folder that has them.")
G = 9.81
# van der Zee, Mundinger & Kuo (2022) Table 1: height (m), mass (kg)
SUBJ = {1: (1.73, 81.8), 2: (1.64, 57.3), 3: (1.90, 97.5), 4: (1.73, 57.0),
        5: (1.63, 56.7), 6: (1.83, 72.6), 7: (1.89, 86.2), 8: (1.85, 88.6),
        9: (1.78, 77.0)}
# file slot k is the paper's trial k+1 (dataset trial_look_up.xlsx / paper Table 2)
REFERENCE_SLOT = 19                    # 1.25 m/s, preferred step length and width
STEP_WIDTH_SLOTS = {26: 0.10, 27: 0.20, 28: 0.30, 29: 0.40}   # laser-line targets (m), p6 only
KNEE_SHIFT_MM = [0, 5, 10, 15, 20]
BOW_MM = [0, 5, 10, 15, 20]
THETA_AM_DEG = 6.0                     # anatomical vs mechanical femoral axis (~6 deg)
S_GRID = np.linspace(0.0, 1.0, 21)     # 0 = knee centre, 1 = hip centre
I_MID, I_SUB = 10, 16                  # s = 0.5 (mid-shaft), s = 0.8 (~subtrochanteric)

def show(fig):
    display(HTML(f'<img src="{fig_svg(fig)}" style="max-width:100%">'))
    plt.close(fig)

In [3]:
def unit(v):
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

def min_rotation(a, b):
    """[T,3,3] rotation taking unit vectors a -> b by the shortest arc."""
    v = np.cross(a, b); c = np.sum(a * b, axis=-1)
    vx = np.zeros(a.shape[:-1] + (3, 3))
    vx[..., 0, 1], vx[..., 0, 2] = -v[..., 2], v[..., 1]
    vx[..., 1, 0], vx[..., 1, 2] = v[..., 2], -v[..., 0]
    vx[..., 2, 0], vx[..., 2, 1] = -v[..., 1], v[..., 0]
    return np.eye(3) + vx + vx @ vx * (1.0 / (1.0 + c))[..., None, None]

def lateral_dir(kin):
    """[T,3] unit LATERAL direction at the right knee, perpendicular to the
    hip-ankle (mechanical) axis, from the thigh's medio-lateral axis."""
    hip, ankle = kin.prox_pos[:, 2], kin.prox_pos[:, 0]
    a = unit(hip - ankle)
    lat = -kin.r_world[:, 2, :, 1]                 # thigh local y = subject-left = medial
    return unit(lat - a * np.sum(lat * a, axis=-1, keepdims=True))

def varus_shift(kin, d):
    """Move the knee centre laterally by d (m) every frame; hip, ankle, foot and
    the measured ground force stay put; thigh and shank re-posed minimally."""
    knee, hip, ankle = kin.prox_pos[:, 1], kin.prox_pos[:, 2], kin.prox_pos[:, 0]
    knee_new = knee + d * lateral_dir(kin)
    r = kin.r_world.copy()
    r[:, 1] = min_rotation(unit(knee - ankle), unit(knee_new - ankle)) @ kin.r_world[:, 1]
    r[:, 2] = min_rotation(unit(hip - knee), unit(hip - knee_new)) @ kin.r_world[:, 2]
    prox, dist = kin.prox_pos.copy(), kin.dist_pos.copy()
    prox[:, 1] = knee_new; dist[:, 2] = knee_new
    return SegmentKinematics(t=kin.t, rate=kin.rate, r_world=r, prox_pos=prox, dist_pos=dist)

def frontal_hka(kin):
    """[T] frontal hip-knee-ankle deviation (deg, + = varus) from the joint centres."""
    hip, knee, ankle = kin.prox_pos[:, 2], kin.prox_pos[:, 1], kin.prox_pos[:, 0]
    x = kin.r_world[:, 2, :, 0]
    proj = lambda v: unit(v - x * np.sum(v * x, axis=-1, keepdims=True))
    u_t, u_s = proj(hip - knee), proj(knee - ankle)
    return np.degrees(np.arctan2(-np.sum(np.cross(u_s, u_t) * x, axis=-1),
                                 np.sum(u_s * u_t, axis=-1)))

def stance_windows(trial, min_len=20):
    p = io.plate_for_side(trial, "r")
    fz = io.analog_to_mocap(trial, trial.force[:, p, 2])
    _, ev = detect_contact(fz, 20.0, min_gap=5)
    return [(a, b) for a, b in ev if a > 0 and b < len(fz) - 1 and b - a > min_len]

def kam(skel, kin, res, params):
    """External knee adduction moment (N m, + = varus) in the thigh frame."""
    x = chain_kinematics(skel, kin, params.lowpass_hz, params.filter_order)["r"][:, 2, :, 0]
    return -np.sum(res.joint_torque[:, 1] * x, axis=-1)

def shaft_sections(skel, kin, res, params, e_of_s):
    """Net internal wrench at femoral cross-sections s (S_GRID), moved to the
    shaft centroid c(s) = q(s) + e(s)*lateral.  M_lat > 0 = LATERAL cortex in
    tension; P = axial compression."""
    ck = chain_kinematics(skel, kin, params.lowpass_hz, params.filter_order)
    knee, hip, com_t = ck["prox"][:, 1], ck["prox"][:, 2], ck["com"][:, 2]
    s_com = np.median(np.linalg.norm(com_t - knee, axis=1) / np.linalg.norm(hip - knee, axis=1))
    fk, mk = res.joint_force[:, 1], res.joint_torque[:, 1]
    fh, mh = res.joint_force[:, 2], res.joint_torque[:, 2]
    x, y, z = (ck["r"][:, 2, :, k] for k in range(3))
    out = {k: np.zeros((len(knee), len(S_GRID))) for k in ("M_lat", "P")}
    for i, s in enumerate(S_GRID):
        q = knee + s * (hip - knee)
        f_s, m_s = (fk, mk + np.cross(knee - q, fk)) if s < s_com else (fh, mh + np.cross(hip - q, fh))
        c = q + e_of_s(s) * (-y)
        m_c = m_s + np.cross(q - c, f_s)
        out["M_lat"][:, i] = -np.sum(m_c * x, axis=-1)
        out["P"][:, i] = -np.sum(f_s * z, axis=-1)
    return out

def e_profile(L, theta_am_deg, delta0):
    """Lateral offset of the shaft centroid from the hip-knee line (m): the
    anatomical-mechanical angle plus a parabolic bow with apex offset delta0."""
    t = np.tan(np.radians(theta_am_deg))
    return lambda s: s * L * t + 4.0 * delta0 * s * (1.0 - s)

def analyse(trial, subj):
    """KAM vs knee shift, and shaft bending vs bow, averaged over full right stances."""
    params = p1_params()
    skel, kin, ground = io.build_chain(trial, side="r")
    mass = float(skel.mass.sum() / (0.0137 + 0.0433 + 0.1416 + 0.1117))   # de Leva leg+pelvis fractions
    wins = stance_windows(trial)
    L_th = float(np.median(np.linalg.norm(kin.prox_pos[:, 2] - kin.prox_pos[:, 1], axis=1)))
    def norm(sig):
        return np.array([np.interp(np.linspace(0, 1, 101), np.linspace(0, 1, b - a + 1), sig[a:b + 1])
                         for a, b in wins])
    out = {"subj": subj, "mass": mass, "height": SUBJ[subj][0], "n": len(wins),
           "kam": {}, "hka": {}, "shaft": {}}
    for d in KNEE_SHIFT_MM:
        k_d = varus_shift(kin, d / 1000) if d else kin
        res = inverse_dynamics(skel, k_d, ground, params)
        out["kam"][d] = norm(kam(skel, k_d, res, params))
        out["hka"][d] = float(norm(frontal_hka(k_d))[:, 10:50].mean())
        if d == 0:
            res0 = res
    out["fz"] = norm(ground.force[:, 2])
    for th in (0.0, THETA_AM_DEG):
        for b in BOW_MM:
            sec = shaft_sections(skel, kin, res0, params, e_profile(L_th, th, b / 1000))
            ml = np.stack([norm(sec["M_lat"][:, i]) for i in range(len(S_GRID))], -1)
            out["shaft"][(th, b)] = ml
            if th == 0.0 and b == 0:
                out["P"] = np.stack([norm(sec["P"][:, i]) for i in range(len(S_GRID))], -1)
    out["v3d"] = norm(trial.reference["r_kne_moment"][:, 0]) if "r_kne_moment" in trial.reference else None
    return out

def peak1(curves):          # mean over stances of the first-half (loading) peak
    return float(np.mean(curves[:, :50].max(1)))

### p6 at 1.25 m/s: the knee

In [4]:
trials6 = io.load_v3d_trials(DATA.format(6), skip_empty=False)
r6 = analyse(trials6[REFERENCE_SLOT], 6)
m6 = r6["mass"]
pct = np.arange(101)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), layout="constrained")
cols = plt.cm.viridis(np.linspace(0, 0.85, len(KNEE_SHIFT_MM)))
for d, c in zip(KNEE_SHIFT_MM, cols):
    ax[0].plot(pct, r6["kam"][d].mean(0) / m6, color=c, lw=1.8,
               label=f"{d} mm ({r6['hka'][d] - r6['hka'][0]:.1f}° varus)")
ax[0].axhline(0, color="gray", lw=0.6)
style_axes(ax[0], "% stance", "external knee adduction moment (N m/kg)")
ax[0].legend(title="knee moved laterally", fontsize=8)
added = [r6["hka"][d] - r6["hka"][0] for d in KNEE_SHIFT_MM]
ax[1].plot(added, [peak1(r6["kam"][d]) / m6 for d in KNEE_SHIFT_MM], "o-", color=JOINT_COLORS[1], label="first peak")
style_axes(ax[1], "added varus (deg)", "first peak (N m/kg)")
show(fig)

fz1 = peak1(r6["fz"])
rows = ["| knee shift (mm) | added varus (°) | KAM first peak (N m/kg) | (%BW·Ht) | ΔKAM (N m) | d·Fz check (N m) |", "|---|---|---|---|---|---|"]
for d in KNEE_SHIFT_MM:
    k = peak1(r6["kam"][d])
    rows.append(f"| {d} | {r6['hka'][d] - r6['hka'][0]:.2f} | {k / m6:.3f} | {100 * k / (m6 * G * r6['height']):.2f} | "
                f"{k - peak1(r6['kam'][0]):.1f} | {d / 1000 * fz1:.1f} |")
display(HTML("<p>p6, 1.25 m/s, %d right stances, body mass %.1f kg (from force)</p>" % (r6["n"], m6)))
from IPython.display import Markdown
display(Markdown("\n".join(rows)))

| knee shift (mm) | added varus (°) | KAM first peak (N m/kg) | (%BW·Ht) | ΔKAM (N m) | d·Fz check (N m) |
|---|---|---|---|---|---|
| 0 | 0.00 | 0.510 | 2.84 | 0.0 | 0.0 |
| 5 | 1.30 | 0.558 | 3.11 | 3.9 | 4.5 |
| 10 | 2.59 | 0.606 | 3.38 | 7.8 | 9.0 |
| 15 | 3.89 | 0.654 | 3.64 | 11.8 | 13.6 |
| 20 | 5.18 | 0.702 | 3.91 | 15.7 | 18.1 |

The extra moment is close to $d\cdot F_z$ (the last two columns; a bit under, because the ground force isn't purely vertical and the knee moves perpendicular to the leg, not the floor), so the counterfactual is doing what the lever-arm picture says. About +0.037 N m/kg, or 7 %, per degree of varus on p6. As a check that our KAM is the real thing: it tracks Visual3D's own frontal knee moment for this trial (next cell).

In [5]:
ref = trials6[REFERENCE_SLOT].reference["r_kne_moment"]
wins = stance_windows(trials6[REFERENCE_SLOT])
def norm6(sig):
    return np.array([np.interp(np.linspace(0, 1, 101), np.linspace(0, 1, b - a + 1), sig[a:b + 1]) for a, b in wins]).mean(0)
ours = r6["kam"][0].mean(0) / m6
cors = [np.corrcoef(ours, norm6(ref[:, c]))[0, 1] for c in range(3)]
c = int(np.argmax(np.abs(cors)))            # the frontal (ab/adduction) component
v = np.sign(cors[c]) * norm6(ref[:, c])
print(f"Visual3D knee moment component {c} (frontal): correlation {cors[c]:+.3f} "
      f"(the sign is just their convention); first peaks {ours[:50].max():.3f} (ours) vs {v[:50].max():.3f} N m/kg (Visual3D)")

Visual3D knee moment component 1 (frontal): correlation -0.998 (the sign is just their convention); first peaks 0.482 (ours) vs 0.529 N m/kg (Visual3D)


### p6 at 1.25 m/s: the femoral shaft

Left: the peak lateral-tension bending moment along the femur during stance, for bows from 0 to 20 mm. Right: the same thing at mid-shaft over the stance phase. The literature ranges are roughly 4–6 mm for older controls and 10–20 mm for femurs with diaphyseal AFF.

In [6]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), layout="constrained")
cols = plt.cm.magma(np.linspace(0.1, 0.8, len(BOW_MM)))
for b, c in zip(BOW_MM, cols):
    ml = r6["shaft"][(THETA_AM_DEG, b)]
    ax[0].plot(S_GRID, ml.max(1).mean(0) / m6, color=c, lw=1.8, label=f"{b} mm")
    ax[1].plot(pct, ml.mean(0)[:, I_MID] / m6, color=c, lw=1.6)
ax[0].axvspan(0.4, 0.6, color="gray", alpha=0.12)
ax[0].text(0.5, ax[0].get_ylim()[0], "usual AFF apex\n(40-60 %)", ha="center", va="bottom", fontsize=8)
style_axes(ax[0], "position along femur (0 = knee, 1 = hip)", "peak lateral-tension bending (N m/kg)")
ax[0].legend(title="bow apex offset", fontsize=8)
ax[1].axhline(0, color="gray", lw=0.6)
style_axes(ax[1], "% stance", "mid-shaft bending (N m/kg)")
show(fig)

P_mid = float(np.mean(r6["P"].max(1)[:, I_MID]))
rows = ["| bow apex offset (mm) | mid-shaft (N m/kg) | change | s = 0.8, ~subtrochanteric (N m/kg) |", "|---|---|---|---|"]
base = r6["shaft"][(THETA_AM_DEG, 0)].max(1).mean(0)
for b in BOW_MM:
    pk = r6["shaft"][(THETA_AM_DEG, b)].max(1).mean(0)
    rows.append(f"| {b} | {pk[I_MID] / m6:.3f} | {100 * (pk[I_MID] / base[I_MID] - 1):+.1f} % | {pk[I_SUB] / m6:.3f} |")
display(Markdown("\n".join(rows)))
slope = (r6["shaft"][(THETA_AM_DEG, 20)].max(1).mean(0)[I_MID] - base[I_MID]) / 20
print(f"mid-shaft slope {slope:.2f} N m per mm of bow; net axial force P = {P_mid:.0f} N "
      f"({P_mid / (m6 * G):.2f} BW), so P·δ predicts {P_mid / 1000:.2f} N m per mm")

| bow apex offset (mm) | mid-shaft (N m/kg) | change | s = 0.8, ~subtrochanteric (N m/kg) |
|---|---|---|---|
| 0 | 0.995 | +0.0 % | 1.231 |
| 5 | 1.043 | +4.8 % | 1.258 |
| 10 | 1.091 | +9.7 % | 1.284 |
| 15 | 1.139 | +14.5 % | 1.311 |
| 20 | 1.187 | +19.3 % | 1.338 |

mid-shaft slope 0.79 N m per mm of bow; net axial force P = 849 N (1.05 BW), so P·δ predicts 0.85 N m per mm


(The small kink in the left panel near 0.55–0.6 isn't real: it's where the cut passes the thigh's centre of mass, because I treat the thigh as a point mass. It's worth about 0.04 N m/kg.)

The lateral cortex is in tension through essentially all of stance, the bow adds bending fastest at mid-shaft (where the bow apex is, and where Asian AFFs mostly happen), and the slope is close to $P\cdot\delta$. One thing that I think matters a lot: the net axial force here is only about 1 BW, because inverse dynamics doesn't see the muscles squeezing the hip. Measured hip contact forces in walking are well above that (Bergmann 2001), so the *extra* bending a bow adds in a real femur ($P\delta$) is probably 2–3× what I show here. Meanwhile the muscles also cancel much of the *baseline* bending (Duda 1997). Put together, I'd guess bowing matters more in vivo than these net numbers suggest, but I can't show that without a muscle model.

### All nine walkers

Same 1.25 m/s reference trial for everyone, self-selected step width.

In [7]:
allr = {6: r6}
for s in range(1, 10):
    if s != 6:
        allr[s] = analyse(io.load_v3d_trials(DATA.format(s), skip_empty=False)[REFERENCE_SLOT], s)
rows = ["| subject | KAM first peak (N m/kg) | +20 mm knee shift | change per ° varus | mid-shaft bending (N m/kg) | +10 mm bow | change |",
        "|---|---|---|---|---|---|---|"]
per_deg, bow_pct = [], []
for s, r in sorted(allr.items()):
    m = r["mass"]; k0, k20 = peak1(r["kam"][0]), peak1(r["kam"][20])
    deg = r["hka"][20] - r["hka"][0]
    b0 = r["shaft"][(THETA_AM_DEG, 0)].max(1).mean(0)[I_MID]
    b10 = r["shaft"][(THETA_AM_DEG, 10)].max(1).mean(0)[I_MID]
    per_deg.append(100 * (k20 / k0 - 1) / deg); bow_pct.append(100 * (b10 / b0 - 1))
    rows.append(f"| p{s} | {k0 / m:.3f} | {k20 / m:.3f} | {per_deg[-1]:+.1f} % | {b0 / m:.3f} | {b10 / m:.3f} | {bow_pct[-1]:+.1f} % |")
display(Markdown("\n".join(rows)))
print(f"across subjects: KAM +{np.median(per_deg):.1f} % per degree of varus (median, range {min(per_deg):.1f} to {max(per_deg):.1f}); "
      f"mid-shaft bending +{np.median(bow_pct):.1f} % for a 10 mm bow (range {min(bow_pct):.1f} to {max(bow_pct):.1f})")

| subject | KAM first peak (N m/kg) | +20 mm knee shift | change per ° varus | mid-shaft bending (N m/kg) | +10 mm bow | change |
|---|---|---|---|---|---|---|
| p1 | 0.713 | 0.946 | +5.6 % | 0.967 | 1.079 | +11.6 % |
| p2 | 0.450 | 0.647 | +7.3 % | 0.870 | 0.967 | +11.1 % |
| p3 | 0.571 | 0.767 | +6.7 % | 1.023 | 1.119 | +9.4 % |
| p4 | 0.510 | 0.744 | +7.6 % | 0.934 | 1.049 | +12.2 % |
| p5 | 0.534 | 0.742 | +6.2 % | 0.811 | 0.914 | +12.8 % |
| p6 | 0.510 | 0.702 | +7.3 % | 0.995 | 1.091 | +9.7 % |
| p7 | 0.439 | 0.639 | +8.8 % | 0.970 | 1.068 | +10.1 % |
| p8 | 0.445 | 0.659 | +8.5 % | 1.018 | 1.123 | +10.3 % |
| p9 | 0.362 | 0.593 | +10.8 % | 0.812 | 0.925 | +13.9 % |

across subjects: KAM +7.3 % per degree of varus (median, range 5.6 to 10.8); mid-shaft bending +11.1 % for a 10 mm bow (range 9.4 to 13.9)


## 3. Could anything help? Wider steps, and some speculation

### Wider steps, on p6

The dataset had step-width conditions (walking on laser lines 0–40 cm apart). In the local export only p6 has them, and the 0 cm trial is unusable (the right plate picks up the left foot about half the time), so this is one person at four widths plus their two self-selected trials. Measured widths, not the targets.

In [8]:
def heel_strikes(tr, side):
    p = io.plate_for_side(tr, side)
    _, ev = detect_contact(io.analog_to_mocap(tr, tr.force[:, p, 2]), 20.0, min_gap=5)
    return np.array([a for a, b in ev if a > 0 and b - a > 10])

def step_width(tr):
    """Mean medio-lateral (lab x) distance between each right heel strike's
    heel marker and the nearest-in-time left heel strike's heel marker."""
    r_on, l_on = heel_strikes(tr, "r"), heel_strikes(tr, "l")
    xr, xl = tr.markers["RCAL"][r_on, 0], tr.markers["LCAL"][l_on, 0]
    return float(np.mean([abs(xr[i] - xl[np.argmin(np.abs(l_on - r_on[i]))]) for i in range(len(r_on))]))

widths = {slot: step_width(trials6[slot]) for slot in (19, 21, 26, 27, 28, 29)}
sw = {slot: (r6 if slot == REFERENCE_SLOT else analyse(trials6[slot], 6)) for slot in widths}
order = sorted(widths, key=widths.get)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), layout="constrained")
cols = plt.cm.cividis(np.linspace(0, 0.9, len(order)))
for slot, c in zip(order, cols):
    ax[0].plot(pct, sw[slot]["kam"][0].mean(0) / sw[slot]["mass"], color=c, lw=1.6, label=f"{100 * widths[slot]:.0f} cm")
style_axes(ax[0], "% stance", "external knee adduction moment (N m/kg)")
ax[0].legend(title="measured step width", fontsize=8)
x = [100 * widths[s] for s in order]
ax[1].plot(x, [peak1(sw[s]["kam"][0]) / sw[s]["mass"] for s in order], "o-", color=JOINT_COLORS[1], label="KAM first peak")
ax[1].plot(x, [sw[s]["shaft"][(THETA_AM_DEG, 0)].max(1).mean(0)[I_MID] / sw[s]["mass"] for s in order], "s-",
           color=JOINT_COLORS[0], label="mid-shaft bending")
style_axes(ax[1], "measured step width (cm)", "peak (N m/kg)")
ax[1].legend(fontsize=8)
show(fig)
rows = ["| measured width (cm) | KAM first peak (N m/kg) | mid-shaft bending (N m/kg) |", "|---|---|---|"]
for s in order:
    rows.append(f"| {100 * widths[s]:.1f} | {peak1(sw[s]['kam'][0]) / sw[s]['mass']:.3f} | "
                f"{sw[s]['shaft'][(THETA_AM_DEG, 0)].max(1).mean(0)[I_MID] / sw[s]['mass']:.3f} |")
display(Markdown("\n".join(rows)))

| measured width (cm) | KAM first peak (N m/kg) | mid-shaft bending (N m/kg) |
|---|---|---|
| 10.8 | 0.492 | 1.005 |
| 15.8 | 0.469 | 0.942 |
| 16.4 | 0.510 | 0.995 |
| 16.6 | 0.489 | 0.975 |
| 24.1 | 0.477 | 0.954 |
| 34.6 | 0.440 | 0.916 |

Going from p6's own width (16.4 and 16.6 cm in the two self-selected trials) to 34.6 cm lowers the knee adduction moment first peak by 10–14 % and mid-shaft bending by 6–8 %. Compared with section 2, that's about the same as taking 6–8 mm off the bow. But look at the narrow end: the 15.8 cm trial comes out *lower* than the self-selected ones at almost the same width, so the run-to-run scatter is about as big as the step between neighbouring widths. Only the widest condition clearly stands out. And the curves on the left show *where* in stance width helps: mostly the second half (push-off), not the first peak, which barely moves until the widest step. It's a modest effect, from one person, walking at widths they were told to, for five strides each. But it goes the right way, and it lines up with the simulation result that a wide base is the most effective way to cut hip contact force (Wesseling 2015).

### Some speculation (and I mean speculation)

I don't know if anything here would actually prevent fractures. But here is how I'd think about it, roughly in order of how much I believe it:

1. **Find the bowed femurs before they break.** The bow is acquired and measurable on an ordinary standing radiograph, the fracture sits at its apex, and a bow above about 5° marks the diaphyseal type (Yoo 2017). Screening older Asian women on long-term bisphosphonates for lateral bow, and taking drug holidays seriously in the bowed ones, uses evidence that already exists (drug holidays: Schilcher 2011, Black 2020). This isn't biomechanics, but it's probably the biggest lever.
2. **Keep the tension band.** If the iliotibial band and lateral thigh muscles carry part of the lateral load (Duda 1998, Shin 2017), then strength training of the hip abductors and lateral thigh might shift bending back towards compression. That's testable with a muscle-driven model on exactly these walkers.
3. **Walk a bit wider, or sway a bit.** A wider base helped p6 by an amount comparable to several mm of bow. Lateral trunk sway cut frontal-plane hip and knee moments a lot in Mündermann 2008, but it costs energy and looks odd, and it's tied up with step width (Anderson 2018). A cane in the opposite hand cuts frontal-plane loads (Kemp 2008, Chan 2005), and it might be the most acceptable version for someone who already has prodromal thigh pain.
4. **Shoes and insoles.** Lateral wedges reduce the knee adduction moment a little (Kerrigan 2002, Hinman 2008), but I couldn't find any evidence they change femoral loading. I'd put these last.
5. **What I'd do next.** Run a muscle-driven model (static optimization is enough) on these walkers with a bowed femur and a curved-beam stress calculation, so the tension-band effect and the in-vivo axial force are both in; and get the raw c3d step-width trials for all subjects (they're on figshare) through our c3d pipeline, so the wide-step result rests on more than one person.

### Caveats

* Net joint loads only, no muscles (see section 2). These are bending *moments*; I don't give stresses because that needs each person's cortical geometry.
* Same gait, different skeleton. People with bowed legs adapt how they walk.
* The bow is modelled as a smooth parabola in the frontal plane only; the anterior bow is ignored.
* Body mass is estimated from the force plates (p6: 82.0 kg here vs 72.6 kg in the paper), which scales everything in N m/kg.
* The step-width result is one subject.

## References

Citation counts from Semantic Scholar, retrieved 2026-10-23. All DOIs resolved.

**Atypical femoral fracture: definition, risk, mechanics**

* Shane E, et al. (2010). Atypical subtrochanteric and diaphyseal femoral fractures: report of a task force of the ASBMR. *J Bone Miner Res* 25:2267–2294. doi:10.1002/jbmr.253 — 1205 citations
* Shane E, et al. (2014). Atypical subtrochanteric and diaphyseal femoral fractures: second report of a task force of the ASBMR. *J Bone Miner Res* 29:1–23. doi:10.1002/jbmr.1998 — 1745
* Schilcher J, Michaëlsson K, Aspenberg P (2011). Bisphosphonate use and atypical fractures of the femoral shaft. *N Engl J Med* 364:1728–1737. doi:10.1056/NEJMoa1010650 — 623
* Schilcher J, Koeppen V, Ranstam J, et al. (2013). Atypical femoral fractures are a separate entity, characterized by highly specific radiographic features. *Bone* 52:389–392. doi:10.1016/j.bone.2012.10.016 — 76
* Black DM, Geiger EJ, Eastell R, et al. (2020). Atypical femur fracture risk versus fragility fracture prevention with bisphosphonates. *N Engl J Med* 383:743–753. doi:10.1056/NEJMoa1916525 — 324
* Black DM, Condra K, Adams AL, Eastell R (2022). Bisphosphonates and the risk of atypical femur fractures. *Bone* 156:116297. doi:10.1016/j.bone.2021.116297 — 39
* Lo JC, Hui RL, Grimsrud CD, et al. (2016). The association of race/ethnicity and risk of atypical femur fracture among older women receiving oral bisphosphonate therapy. *Bone* 85:142–147. doi:10.1016/j.bone.2016.01.002 — 101
* Martelli S, Pivonka P, Ebeling PR (2014). Femoral shaft strains during daily activities: implications for atypical femoral fractures. *Clin Biomech* 29:869–876. doi:10.1016/j.clinbiomech.2014.08.001 — 80
* Haider IT, Schneider P, Michalski A, Edwards WB (2018). Influence of geometry on proximal femoral shaft strains: implications for atypical femoral fracture. *Bone* 110:295–303. doi:10.1016/j.bone.2018.02.015 — 47
* Hagen JE, et al. (2014). Association of atypical femoral fractures with bisphosphonate use by patients with varus hip geometry. *J Bone Joint Surg Am* 96:1905–1909 — 78

**Femoral bowing, alignment and AFF site**

* Sasaki S, Miyakoshi N, Hongo M, et al. (2012). Low-energy diaphyseal femoral fractures associated with bisphosphonate use and severe curved femur: a case series. *J Bone Miner Metab* 30:561–567. doi:10.1007/s00774-012-0358-0 — 132
* Oh Y, Wakabayashi Y, Kurosa Y, Ishizuki M, Okawa A (2014a). Stress fracture of the bowed femoral shaft is another cause of atypical femoral fracture in elderly Japanese: a case series. *J Orthop Sci* 19:579–586
* Oh Y, Wakabayashi Y, Kurosa Y, Fujita K, Okawa A (2014b). Potential pathogenic mechanism for stress fractures of the bowed femoral shaft in the elderly: mechanical analysis by the CT-based finite element method. *Injury* 45:1764–1771. doi:10.1016/j.injury.2014.08.037 — 72
* Oh Y, Fujita K, Wakabayashi Y, Kurosa Y, Okawa A (2017). Location of atypical femoral fracture can be determined by tensile stress distribution influenced by femoral bowing and neck-shaft angle. *Injury* 48:2736–2743. doi:10.1016/j.injury.2017.09.023 — 63
* Saita Y, Ishijima M, Mogami A, et al. (2014). The fracture sites of atypical femoral fractures are associated with the weight-bearing lower limb alignment. *Bone* 66:105–110. doi:10.1016/j.bone.2014.06.008 — 86
* Yoo H, Cho Y, Park Y, Ha S (2017). Lateral femoral bowing and the location of atypical femoral fractures. *Hip Pelvis* 29:127–132. doi:10.5371/hp.2017.29.2.127 — 39
* Jang SP, Yeo I, So SY, et al. (2017). Atypical femoral shaft fractures in female bisphosphonate users were associated with an increased anterolateral femoral bow and a thicker lateral cortex: a case-control study. *Biomed Res Int* 2017:5932496. doi:10.1155/2017/5932496 — 21
* Park YC, Yoon SP, Yang KH (2019). Localization of atypical femoral fracture on straight and bowed femurs. *J Bone Metab* 26:123–131. doi:10.11005/jbm.2019.26.2.123 — 17
* Shin WC, Moon NH, Jang JH, Park KY, Suh KT (2017). Anterolateral femoral bowing and loss of thigh muscle are associated with occurrence of atypical femoral fracture: effect of failed tension band mechanism in mid-thigh. *J Orthop Sci* 22:99–104. doi:10.1016/j.jos.2016.09.009 — 21
* Hyodo K, Nishino T, Kamada H, et al. (2017). Location of fractures and the characteristics of patients with atypical femoral fractures: analyses of 38 Japanese cases. *J Bone Miner Metab*

**Lower-limb geometry in East and Southeast Asian populations**

* Bellemans J, Colyn W, Vandenneucker H, Victor J (2012). Is neutral mechanical alignment normal for all patients? The concept of constitutional varus. *Clin Orthop Relat Res* 470:45–53. doi:10.1007/s11999-011-1936-5 — 1046
* Lasam MP, Lee KJ, Chang CB, et al. (2013). Femoral lateral bowing and varus condylar orientation are prevalent and affect axial alignment of TKA in Koreans. *Clin Orthop Relat Res*. doi:10.1007/s11999-012-2618-7 — 100
* Song MH, Yoo SH, Kang SW, et al. (2015). Coronal alignment of the lower limb and the incidence of constitutional varus knee in Korean females. *Knee Surg Relat Res* 27:49–55. doi:10.5792/ksrr.2015.27.1.49 — 85
* Shimosawa H, Nagura T, Harato K, et al. (2019). Variation of three-dimensional femoral bowing and its relation to physical status and bone mineral density: a study with CT. *Surg Radiol Anat*
* Tsuchie H, Miyakoshi N, Kasukawa Y, et al. (2016). Factors related to curved femur in elderly Japanese women. *Ups J Med Sci* 121:170–173. doi:10.1080/03009734.2016.1185200 — 15
* Maratt J, Schilling PL, Holcombe S, et al. (2014). Variation in the femoral bow: a novel high-throughput analysis of 3922 femurs on cross-sectional imaging. *J Orthop Trauma* 28:6–9
* Thiesen DM, Prange F, Berger-Groch J, et al. (2018). Femoral antecurvation—a 3D CT analysis of 1232 adult femurs. *PLoS One* 13:e0204961. doi:10.1371/journal.pone.0204961 — 31
* Wanezaki Y, Suzuki A, Takakubo Y, et al. (2023). Lower limb alignment in healthy Japanese adults. *J Orthop Sci* 28:200–203. doi:10.1016/j.jos.2021.10.016 — 21

**Varus gait, femoral loading and gait modifications**

* Sharma L, Song J, Felson DT, et al. (2001). The role of knee alignment in disease progression and functional decline in knee osteoarthritis. *JAMA* 286:188–195 — 1624
* Hurwitz DE, Ryals AB, Case JP, Block JA, Andriacchi TP (2002). The knee adduction moment during gait in subjects with knee osteoarthritis is more closely correlated with static alignment than radiographic disease severity. *J Orthop Res* 20:101–107 — 595
* Halder A, Kutzner I, Graichen F, et al. (2012). Influence of limb alignment on mediolateral loading in total knee replacement: in vivo measurements in five patients. *J Bone Joint Surg Am*
* Walter JP, D'Lima DD, Colwell CW, Fregly BJ (2010). Decreased knee adduction moment does not guarantee decreased medial contact force during gait. *J Orthop Res* 28:1348–1354 — 376
* Bergmann G, Deuretzbacher G, Heller M, et al. (2001). Hip contact forces and gait patterns from routine activities. *J Biomech* 34:859–871 — 2297
* Duda GN, Schneider E, Chao EYS (1997). Internal forces and moments in the femur during walking. *J Biomech* 30:933–941. doi:10.1016/S0021-9290(97)00057-2 — 301
* Duda GN, Heller M, Albinger J, et al. (1998). Influence of muscle forces on femoral strain distribution. *J Biomech* 31:841–846. doi:10.1016/S0021-9290(98)00080-3 — 362
* Aamodt A, Lund-Larsen J, Eine J, et al. (1997). In vivo measurements show tensile axial strain in the proximal lateral aspect of the human femur. *J Orthop Res* 15:927–931 — 129
* Mündermann A, Asay JL, Mündermann L, Andriacchi TP (2008). Implications of increased medio-lateral trunk sway for ambulatory mechanics. *J Biomech* 41:165–170 — 279
* Simic M, Hinman RS, Wrigley TV, Bennell KL, Hunt MA (2011). Gait modification strategies for altering medial knee joint load: a systematic review. *Arthritis Care Res* 63:405–426 — 284
* Wesseling M, de Groote F, Meyer C, et al. (2015). Gait alterations to effectively reduce hip contact forces. *J Orthop Res* 33:1094–1102. doi:10.1002/jor.22852 — 84
* Shull PB, Shultz R, Silder A, et al. (2013). Toe-in gait reduces the first peak knee adduction moment in patients with medial compartment knee osteoarthritis. *J Biomech*
* Kemp G, Crossley KM, Wrigley TV, Metcalf BR, Hinman RS (2008). Reducing joint loading in medial knee osteoarthritis: shoes and canes. *Arthritis Rheum* 59:609–614
* Chan GN, Smith AW, Kirtley C, Tsang WW (2005). Changes in knee moments with contralateral versus ipsilateral cane usage in females with knee osteoarthritis. *Clin Biomech* 20:396–404
* Kerrigan DC, Lelas JL, Goggins J, et al. (2002). Effectiveness of a lateral-wedge insole on knee varus torque in patients with knee osteoarthritis. *Arch Phys Med Rehabil*
* Hinman RS, Payne C, Metcalf BR, Wrigley TV, Bennell KL (2008). Lateral wedges in knee osteoarthritis: what are their immediate clinical and biomechanical effects and can these predict a three-month clinical outcome? *Arthritis Rheum*
* Anderson J, King S, Przybyla A, Ranganath L, Barton G (2018). Reduction of frontal plane knee load caused by lateral trunk lean depends on step width. *Gait Posture* 61:483–487. doi:10.1016/j.gaitpost.2018.02.022 — 18

**Newer, few citations (for awareness):** Gani LU et al. (2022), Singapore AFF series without bisphosphonates; Hwang et al. (2022), patient-specific varus axis in finite-element AFF site prediction.

**Data:** van der Zee TJ, Mundinger EM, Kuo AD (2022). A biomechanics dataset of healthy human walking at various speeds, step lengths and step widths. *Sci Data* 9:704. doi:10.1038/s41597-022-01817-1